# DR Stage Classification & Lesion Segmentation — v5 (cache & seg fixes)
**Runs as-is on Kaggle with IDRiD. IMPORTANT: do Run > Restart & Run All — do not re-run cells piecemeal.**

**Why v4 showed val_acc 0.66 but test_acc 0.20:** the test `tf.data` pipeline was cached and got out of sync with train when cells were re-run individually (stale oversampled cache + train/test preprocessing mismatch). A model that reaches val 0.66 cannot truly be at test 0.20 on same-distribution data — the gap was a pipeline artifact, not the model.

**Fixes in v5:** (1) NO `.cache()` on any pipeline — eliminates stale-cache bugs across re-runs. (2) One single `load_fundus` used identically for train/val/test. (3) Segmentation split rebuilt with a positive-pixel diagnostic so empty/misaligned masks are caught. (4) Must Restart & Run All.

**Honest ceiling unchanged:** IDRiD-only 5-class ~50–65%; 20-image 'Mild' stays weak. APTOS is the publication-grade route for grading; keep IDRiD for segmentation.

## 0. Setup

In [ ]:
import os, glob, json, random, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, cohen_kappa_score,
                             f1_score, precision_score, recall_score, accuracy_score, roc_auc_score)
SEED=42; random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
for g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))


: 

## 1. Config

In [ ]:
# >>> If you re-ran cells out of order before, do Run > Restart & Run All now. <<<
DATA_ROOT="/kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset"
OUT="/kaggle/working"; os.makedirs(OUT, exist_ok=True)
CLS_IMG=224; CLS_BS=16; USE_CLAHE=False; WARMUP_EP=25; FT_EP=25; FT_LR=1e-5
NUM_CLASSES=5; CLASS_NAMES=["No DR","Mild","Moderate","Severe","Proliferative"]
SEG_IMG=384; SEG_BS=4; SEG_EPOCHS=60
print("config ok | DATA_ROOT exists:", os.path.isdir(DATA_ROOT))


## 2. File indexing

In [ ]:
def index_images(root):
    idx={}
    for p in glob.glob(os.path.join(root,"**","*.jpg"),recursive=True):
        idx.setdefault(os.path.splitext(os.path.basename(p))[0], p)
    return idx
ALL_JPG=index_images(DATA_ROOT)
grading_paths={k:v for k,v in ALL_JPG.items() if "Disease Grading" in v}
seg_paths   ={k:v for k,v in ALL_JPG.items() if "Segmentation"   in v}
print("grading:", len(grading_paths), "| segmentation:", len(seg_paths))


## 3. Labels, split, oversample

In [ ]:
def find_csv(sub):
    return [p for p in glob.glob(os.path.join(DATA_ROOT,"**","*.csv"),recursive=True)
            if "grading" in p.lower() and sub in p.lower()]
train_csv=find_csv("train"); test_csv=find_csv("test")
def load_grades(csv):
    df=pd.read_csv(csv); df=df.loc[:,~df.columns.str.contains("^Unnamed")].iloc[:,:2]
    df.columns=["image_id","grade"]; df["image_id"]=df["image_id"].astype(str).str.strip()
    df=df.dropna(subset=["grade"]); df["grade"]=df["grade"].astype(int)
    df["path"]=df["image_id"].map(grading_paths)
    return df.dropna(subset=["path"]).reset_index(drop=True)
df_full=load_grades(train_csv[0]); df_test=load_grades(test_csv[0])
df_tr,df_val=train_test_split(df_full,test_size=0.15,stratify=df_full["grade"],random_state=SEED)
df_tr,df_val=df_tr.reset_index(drop=True),df_val.reset_index(drop=True)
def oversample(df):
    n=df["grade"].value_counts().max()
    return pd.concat([s.sample(n,replace=True,random_state=SEED) for _,s in df.groupby("grade")]
                    ).sample(frac=1,random_state=SEED).reset_index(drop=True)
df_tr_bal=oversample(df_tr)
print("train/val/test:",len(df_tr),len(df_val),len(df_test),"| balanced train:",len(df_tr_bal))

# ---- DIAGNOSTIC: sanity baselines ----
MAJ=df_test["grade"].value_counts(normalize=True).max()
print(f"\n[DIAGNOSTIC] majority-class baseline on test = {MAJ:.3f}  (model must beat this)")
print("[DIAGNOSTIC] test label distribution:", np.bincount(df_test['grade'],minlength=5).tolist())


## 4. Preprocessing (circle-crop + CLAHE) + sanity view of labels

In [ ]:
def circle_crop(img,tol=7):
    g=cv2.cvtColor(img,cv2.COLOR_RGB2GRAY); m=g>tol
    if m.sum()==0: return img
    c=np.argwhere(m); y0,x0=c.min(0); y1,x1=c.max(0)+1; return img[y0:y1,x0:x1]
def clahe_rgb(img):
    lab=cv2.cvtColor(img,cv2.COLOR_RGB2LAB); l,a,b=cv2.split(lab)
    l=cv2.createCLAHE(2.0,(8,8)).apply(l); return cv2.cvtColor(cv2.merge((l,a,b)),cv2.COLOR_LAB2RGB)
def load_fundus(path,size):
    img=cv2.cvtColor(cv2.imread(path),cv2.COLOR_BGR2RGB)
    img=cv2.resize(circle_crop(img),(size,size),interpolation=cv2.INTER_AREA)
    if USE_CLAHE: img=clahe_rgb(img)
    return img.astype("float32")

# DIAGNOSTIC: show 2 images per class so you can confirm images<->labels are aligned
fig,ax=plt.subplots(NUM_CLASSES,2,figsize=(6,3*NUM_CLASSES))
for c in range(NUM_CLASSES):
    sub=df_tr[df_tr["grade"]==c]["path"].head(2).tolist()
    for j in range(2):
        ax[c,j].axis("off")
        if j<len(sub):
            ax[c,j].imshow(load_fundus(sub[j],CLS_IMG).astype("uint8"))
            ax[c,j].set_title(CLASS_NAMES[c])
plt.tight_layout(); plt.savefig(f"{OUT}/fig_label_sanity.png",dpi=150,bbox_inches="tight"); plt.show()


## 5. Pipelines + augmentation

In [ ]:
AUTOTUNE=tf.data.AUTOTUNE
def _load(path,size): return load_fundus(path.numpy().decode(),int(size))
def make_ds(df,size,bs,training):
    ds=tf.data.Dataset.from_tensor_slices((df["path"].values,df["grade"].values.astype("int32")))
    def _m(p,y):
        img=tf.py_function(_load,[p,size],tf.float32); img.set_shape([size,size,3]); return img,y
    ds=ds.map(_m,num_parallel_calls=AUTOTUNE)
    if training: ds=ds.shuffle(len(df),seed=SEED,reshuffle_each_iteration=True)
    return ds.batch(bs).prefetch(AUTOTUNE)
ds_tr=make_ds(df_tr,CLS_IMG,CLS_BS,True)   # natural distribution (no oversampling)
ds_val=make_ds(df_val,CLS_IMG,CLS_BS,False)
ds_te=make_ds(df_test,CLS_IMG,CLS_BS,False)
aug=models.Sequential([layers.RandomFlip("horizontal_and_vertical"),
                       layers.RandomRotation(0.08,fill_mode="constant"),
                       layers.RandomZoom(0.1,fill_mode="constant")],name="augment")
print("pipelines ready")


## 6. Models

In [ ]:
mbv2_pre=tf.keras.applications.mobilenet_v2.preprocess_input
def build_model(kind,size=CLS_IMG,n=NUM_CLASSES):
    inp=layers.Input((size,size,3)); x=aug(inp)
    if kind=="EfficientNetB0":
        base=tf.keras.applications.EfficientNetB0(include_top=False,weights="imagenet",input_shape=(size,size,3)); x=base(x)
    elif kind=="MobileNetV2":
        x=layers.Lambda(mbv2_pre)(x)
        base=tf.keras.applications.MobileNetV2(include_top=False,weights="imagenet",alpha=1.0,input_shape=(size,size,3)); x=base(x)
    elif kind=="Lightweight_MobileNetV2":
        x=layers.Lambda(mbv2_pre)(x)
        base=tf.keras.applications.MobileNetV2(include_top=False,weights="imagenet",alpha=0.35,input_shape=(size,size,3)); x=base(x)
    x=layers.GlobalAveragePooling2D()(x); x=layers.Dropout(0.4)(x)
    x=layers.Dense(128,activation="relu")(x); x=layers.Dropout(0.3)(x)
    out=layers.Dense(n,activation="softmax")(x)
    m=models.Model(inp,out,name=kind); m.base=base; return m
print("builder ready")


## 7. Train: head warm-up -> fine-tune with **BatchNorm kept frozen**

In [ ]:
def cbs(tag):
    return [callbacks.ModelCheckpoint(f"{OUT}/{tag}_best.keras",monitor="val_accuracy",mode="max",save_best_only=True),
            callbacks.EarlyStopping(monitor="val_accuracy",mode="max",patience=12,restore_best_weights=True),
            callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=5,min_lr=1e-7)]
def train(kind):
    m=build_model(kind)
    # phase 1: frozen backbone, train head
    m.base.trainable=False
    m.compile(optimizers.Adam(1e-3),"sparse_categorical_crossentropy",metrics=["accuracy"])
    h1=m.fit(ds_tr,validation_data=ds_val,epochs=WARMUP_EP,callbacks=cbs(kind),verbose=2).history
    # phase 2: unfreeze conv, KEEP BatchNorm frozen, low LR
    m.base.trainable=True
    nbn=0
    for l in m.base.layers:
        if isinstance(l,layers.BatchNormalization): l.trainable=False; nbn+=1
    print(f"[{kind}] fine-tune: BatchNorm layers kept frozen = {nbn}")
    m.compile(optimizers.Adam(FT_LR),"sparse_categorical_crossentropy",metrics=["accuracy"])
    h2=m.fit(ds_tr,validation_data=ds_val,epochs=FT_EP,callbacks=cbs(kind),verbose=2).history
    return m,{k:h1.get(k,[])+h2.get(k,[]) for k in h2}

def evaluate_cls(m,tag):
    yt=np.concatenate([y.numpy() for _,y in ds_te]); pr=m.predict(ds_te,verbose=0); yp=pr.argmax(1)
    r={"model":tag,"accuracy":accuracy_score(yt,yp),
       "precision":precision_score(yt,yp,average="macro",zero_division=0),
       "recall":recall_score(yt,yp,average="macro",zero_division=0),
       "f1":f1_score(yt,yp,average="macro",zero_division=0),
       "qwk":cohen_kappa_score(yt,yp,weights="quadratic")}
    try: r["auc_ovr"]=roc_auc_score(yt,pr,multi_class="ovr",labels=np.arange(NUM_CLASSES),average="macro")
    except Exception: r["auc_ovr"]=float("nan")
    print(f"\n=== {tag} ==="); [print(f"{k:>10}: {v:.4f}") for k,v in r.items() if k!="model"]
    print("[DIAGNOSTIC] pred distribution:", np.bincount(yp,minlength=5).tolist(),
          "| if this is one number, the model collapsed")
    print(classification_report(yt,yp,target_names=CLASS_NAMES,zero_division=0))
    return r,yt,yp
def plot_hist(h,tag):
    fig,ax=plt.subplots(1,2,figsize=(11,4))
    ax[0].plot(h.get("accuracy",[]),label="train"); ax[0].plot(h.get("val_accuracy",[]),label="val"); ax[0].set_title(f"{tag} acc"); ax[0].legend()
    ax[1].plot(h.get("loss",[]),label="train"); ax[1].plot(h.get("val_loss",[]),label="val"); ax[1].set_title(f"{tag} loss"); ax[1].legend()
    plt.tight_layout(); plt.savefig(f"{OUT}/fig_history_{tag}.png",dpi=200,bbox_inches="tight"); plt.show()
def plot_cm(yt,yp,tag):
    cm=confusion_matrix(yt,yp,labels=np.arange(NUM_CLASSES)); plt.figure(figsize=(5.5,4.5))
    plt.imshow(cm,cmap="Blues"); plt.title(f"{tag} confusion matrix"); plt.colorbar()
    plt.xticks(range(5),CLASS_NAMES,rotation=45,ha="right"); plt.yticks(range(5),CLASS_NAMES)
    for i in range(5):
        for j in range(5): plt.text(j,i,cm[i,j],ha="center",color="white" if cm[i,j]>cm.max()/2 else "black")
    plt.ylabel("true"); plt.xlabel("pred"); plt.tight_layout()
    plt.savefig(f"{OUT}/fig_cm_{tag}.png",dpi=200,bbox_inches="tight"); plt.show()


### 7a. Quick single-model check (RUN THIS FIRST)
Train only EfficientNetB0 and confirm it beats the majority baseline before committing to all three.

In [ ]:
m0,h0=train("EfficientNetB0")
m0.save(f"{OUT}/dr_classifier.h5"); print("Exported dr_classifier.h5")
plot_hist(h0,"EfficientNetB0")
r0,yt0,yp0=evaluate_cls(m0,"EfficientNetB0"); plot_cm(yt0,yp0,"EfficientNetB0")
print("\n>>> Beats majority baseline?" , r0["accuracy"]>MAJ, f"(acc={r0['accuracy']:.3f} vs base={MAJ:.3f})")


### 7b. Train the remaining two models

In [ ]:
results=[r0]; trained={"EfficientNetB0":m0}
for kind in ["MobileNetV2","Lightweight_MobileNetV2"]:
    print("\n############### TRAINING",kind,"###############")
    m,h=train(kind); trained[kind]=m
    plot_hist(h,kind); r,yt,yp=evaluate_cls(m,kind); results.append(r); plot_cm(yt,yp,kind)


## 8. Comparison table + chart

In [ ]:
res=pd.DataFrame(results)
res["params_M"]=[trained[k].count_params()/1e6 for k in res["model"]]
res=res[["model","accuracy","precision","recall","f1","qwk","auc_ovr","params_M"]]
res.to_csv(f"{OUT}/classification_results.csv",index=False); display(res.round(4))
x=np.arange(len(res)); w=0.2; plt.figure(figsize=(9,5))
for i,mt in enumerate(["accuracy","precision","recall","f1"]): plt.bar(x+i*w,res[mt]*100,w,label=mt)
plt.xticks(x+1.5*w,res["model"],rotation=10); plt.ylim(0,105); plt.ylabel("%")
plt.title("Performance Comparison of Models"); plt.legend(); plt.grid(axis="y",ls="--",alpha=.4)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_model_comparison.png",dpi=200,bbox_inches="tight"); plt.show()


## 9. Segmentation data — binary lesion masks

In [ ]:
def index_masks(root):
    out={}
    for p in glob.glob(os.path.join(root,"**","*.tif"),recursive=True):
        if "Segmentation" not in p or "Optic Disc" in p or "_OD" in os.path.basename(p): continue
        out.setdefault("_".join(os.path.basename(p).split("_")[:2]),[]).append(p)
    return out
MASK_IDX=index_masks(DATA_ROOT)
def seg_is_train(k):
    return ("Training" in seg_paths.get(k,"")) or ("a. Training" in seg_paths.get(k,""))
seg_tr_keys=[k for k in MASK_IDX if k in seg_paths and seg_is_train(k)]
seg_te_keys=[k for k in MASK_IDX if k in seg_paths and not seg_is_train(k)]
print("seg train:",len(seg_tr_keys),"| seg test:",len(seg_te_keys))
def build_mask(key,size):
    m=np.zeros((size,size),np.float32)
    for mp in MASK_IDX[key]:
        g=cv2.imread(mp,cv2.IMREAD_GRAYSCALE)
        if g is None: continue
        g=cv2.resize(g,(size,size),interpolation=cv2.INTER_NEAREST)
        m=np.maximum(m,(g>0).astype("float32"))
    return m[...,None]
# DIAGNOSTIC: positive-pixel fraction per split (must be >0 for both)
def pos_frac(keys):
    fr=[build_mask(k,256).mean() for k in keys[:min(len(keys),20)]]
    return float(np.mean(fr)) if fr else 0.0
print(f"[DIAGNOSTIC] mean lesion-pixel fraction  train={pos_frac(seg_tr_keys):.5f}  test={pos_frac(seg_te_keys):.5f}")
print("   (if either is ~0, masks are empty/misaligned -> fix paths before training)")
def _seg(key,size):
    k=key.numpy().decode(); s=int(size); return load_fundus(seg_paths[k],s), build_mask(k,s)
def make_seg(keys,size,bs,training):
    ds=tf.data.Dataset.from_tensor_slices(keys)
    def _m(k):
        img,msk=tf.py_function(_seg,[k,size],[tf.float32,tf.float32])
        img.set_shape([size,size,3]); msk.set_shape([size,size,1]); return img,msk
    ds=ds.map(_m,num_parallel_calls=AUTOTUNE)
    if training:
        def _a(img,msk):
            k=tf.random.uniform([],0,4,tf.int32); img=tf.image.rot90(img,k); msk=tf.image.rot90(msk,k)
            if tf.random.uniform([])>.5: img=tf.image.flip_left_right(img); msk=tf.image.flip_left_right(msk)
            if tf.random.uniform([])>.5: img=tf.image.flip_up_down(img); msk=tf.image.flip_up_down(msk)
            return img,msk
        ds=ds.shuffle(len(keys),seed=SEED).map(_a,num_parallel_calls=AUTOTUNE)
    return ds.batch(bs).prefetch(AUTOTUNE)
# segmentation uses CLAHE ON (helps lesion contrast) regardless of classification setting
USE_CLAHE=True
seg_tr=make_seg(seg_tr_keys,SEG_IMG,SEG_BS,True); seg_te=make_seg(seg_te_keys,SEG_IMG,SEG_BS,False)
print("segmentation pipelines ready")

## 10. Attention Efficient U-Net + Dice/Focal

In [ ]:
def attn(skip,g,inter):
    a=layers.Activation("relu")(layers.add([layers.Conv2D(inter,1,padding="same")(g),
                                            layers.Conv2D(inter,1,padding="same")(skip)]))
    return layers.multiply([skip,layers.Conv2D(1,1,padding="same",activation="sigmoid")(a)])
def up(x,skip,f):
    x=layers.UpSampling2D(2,interpolation="bilinear")(x); skip=attn(skip,x,f//2)
    x=layers.Concatenate()([x,skip])
    for _ in range(2):
        x=layers.SeparableConv2D(f,3,padding="same",use_bias=False)(x); x=layers.BatchNormalization()(x); x=layers.Activation("relu")(x)
    return x
def build_unet(size=SEG_IMG):
    inp=layers.Input((size,size,3)); x=layers.Lambda(mbv2_pre)(inp)
    enc=tf.keras.applications.MobileNetV2(include_top=False,weights="imagenet",input_shape=(size,size,3))
    names=["block_1_expand_relu","block_3_expand_relu","block_6_expand_relu","block_13_expand_relu"]
    encoder=models.Model(enc.input,[enc.get_layer(n).output for n in names]+[enc.get_layer("out_relu").output])
    s1,s2,s3,s4,b=encoder(x); d=up(b,s4,256); d=up(d,s3,128); d=up(d,s2,64); d=up(d,s1,32)
    d=layers.UpSampling2D(2,interpolation="bilinear")(d); d=layers.SeparableConv2D(16,3,padding="same",activation="relu")(d)
    return models.Model(inp,layers.Conv2D(1,1,activation="sigmoid")(d),name="Attention_Efficient_UNet")
unet=build_unet(); print("U-Net params: %.2fM"%(unet.count_params()/1e6))
def dice_coef(y,p,s=1.):
    y=tf.reshape(y,[-1]); p=tf.reshape(p,[-1]); return (2*tf.reduce_sum(y*p)+s)/(tf.reduce_sum(y)+tf.reduce_sum(p)+s)
def iou_metric(y,p,s=1.):
    y=tf.reshape(y,[-1]); p=tf.reshape(tf.cast(p>0.5,tf.float32),[-1]); inter=tf.reduce_sum(y*p)
    return (inter+s)/(tf.reduce_sum(y)+tf.reduce_sum(p)-inter+s)
def focal(y,p,a=.8,g=2.):
    p=tf.clip_by_value(p,1e-6,1-1e-6); bce=-(y*tf.math.log(p)+(1-y)*tf.math.log(1-p))
    return tf.reduce_mean((y*a+(1-y)*(1-a))*tf.pow(1-(y*p+(1-y)*(1-p)),g)*bce)
def combo(y,p): return 1-dice_coef(y,p)+focal(y,p)
unet.compile(optimizers.Adam(1e-3),combo,metrics=[dice_coef,iou_metric])
seg_cbs=[callbacks.ModelCheckpoint(f"{OUT}/unet_best.keras",monitor="val_dice_coef",mode="max",save_best_only=True),
         callbacks.EarlyStopping(monitor="val_dice_coef",mode="max",patience=12,restore_best_weights=True),
         callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=5,min_lr=1e-6)]
h_seg=unet.fit(seg_tr,validation_data=seg_te,epochs=SEG_EPOCHS,callbacks=seg_cbs,verbose=2).history
unet.save(f"{OUT}/dr_segmenter.h5"); print("Exported dr_segmenter.h5")


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(h_seg["dice_coef"],label="train"); ax[0].plot(h_seg["val_dice_coef"],label="val"); ax[0].set_title("Dice"); ax[0].legend()
ax[1].plot(h_seg["loss"],label="train"); ax[1].plot(h_seg["val_loss"],label="val"); ax[1].set_title("Loss"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/fig_seg_curves.png",dpi=200,bbox_inches="tight"); plt.show()
ev=unet.evaluate(seg_te,verbose=0); seg_m={"loss":ev[0],"dice":ev[1],"iou":ev[2]}
print("Segmentation test:",{k:round(v,4) for k,v in seg_m.items()})
json.dump(seg_m,open(f"{OUT}/segmentation_results.json","w"),indent=2)
n=min(4,len(seg_te_keys)); fig,ax=plt.subplots(n,3,figsize=(10,3*n))
for i,k in enumerate(seg_te_keys[:n]):
    img=load_fundus(seg_paths[k],SEG_IMG); gt=build_mask(k,SEG_IMG)[:,:,0]; pr=unet.predict(img[None],verbose=0)[0,:,:,0]
    ax[i,0].imshow(img.astype("uint8")); ax[i,0].set_title(k); ax[i,0].axis("off")
    ax[i,1].imshow(gt,cmap="gray"); ax[i,1].set_title("GT"); ax[i,1].axis("off")
    ax[i,2].imshow(pr>0.5,cmap="gray"); ax[i,2].set_title("pred"); ax[i,2].axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_seg_predictions.png",dpi=200,bbox_inches="tight"); plt.show()
print("\nArtifacts in /kaggle/working:"); [print(" ",f) for f in sorted(os.listdir(OUT))]
